# 04 — Train Unet2D (masked L1 baseline or adv-NO)

**Active data:** `data/sample_18_21_smallfixwin_uv_dt360` (2018–2021, u+v, 3 cond + 1 pred, `DT=360` h, N≈1461).

**Runs:** each experiment uses a unique `RUN_NAME` (`RUN_BASE` + `RUN_TAG`). Outputs live in `Models/adv_no/{RUN_NAME}/` and are **never overwritten** if that folder already has results.

**Default training (from 30-ep wd ablation):** **100 epochs**, **AdamW**, `weight_decay=3e-4`, val metric **RMSE** (random 70/30 split, seed 42).

`# Old:` `sample_23_*` (2023), `sample_13_22_*` (2013–2022), `sample_16_19_*` dt96 — commented in config cell.

**Active loss:** `LOSS_MODE = "adv_no"` → **L1 + VGG-19 perceptual + adversarial** (`PERCEPTUAL_BACKEND = "vgg"`).

**Loss mode** (`LOSS_MODE`):
- `"l1"` — **masked L1 only** on u,v (or speed); no perceptual net, no discriminator
- `"adv_no"` — **L1 + perceptual + adversarial** with optional staging:
  - `PERCEPTUAL_BACKEND = "vgg"` (multi-layer VGG-19 L1) or `"lpips"` (LPIPS)
  - L1 through `PERCEPTUAL_WARMUP_EPOCHS`, then + perceptual, then + adversarial after `WARMUP_EPOCHS`
  - Set both warmup epochs to `0` to use L1 + VGG + adversarial from epoch 1

**Loss spatial mode** (`LOSS_SPATIAL_MODE`):
- `"full"` — uniform loss over the whole valid field
- `"gaussian_center"` — center-focused loss via Gaussian weight G (set `LOSS_GAUSSIAN_SIGMA`)

**Before training:** optional loss-mask preview (`PREVIEW_LOSS_MASK`) shows uniform G or Gaussian G.

**Reconstruction metric:** `LOSS_METRIC = "components"` (u,v L1) or `"speed"` (L1 on |U,V|).

**Plots** during training: u, v, speed rows over the **full** field (speed is viz-only).

In [1]:
import json
import subprocess
import sys
from dataclasses import asdict
from pathlib import Path

from src.train.adv_no_train import AdvNOTrainConfig, run_adv_no_training

In [2]:
# --- training data prefix (switch here) ---
DT = 360  # 15-day / 360 h between frames (3 cond + 1 pred export)
# DT = 96  # 4-day steps (old: 60-frame export)

DATA_PREFIX = "data/sample_18_21_smallfixwin_uv_dt360"  # train 2018–2021
# DATA_PREFIX = "data/sample_23_smallfixwin_uv_dt360"  # old holdout 2023 only
# DATA_PREFIX = "data/sample_13_22_smallfixwin_uv_dt360"  # old train 2013–2022
# DATA_PREFIX = "data/sample_16_19_smallfixwin_uv_dt96"  # old 2016–2019 dt96

_sims_path = Path(f"{DATA_PREFIX}_sims.npy")
if not _sims_path.exists():
    raise FileNotFoundError(
        f"Missing training export: {_sims_path}\n"
        "Finish 01_export_fix_window_data.ipynb for sample_18_21_smallfixwin_uv_dt360 first."
    )

# Unique name per experiment — change RUN_TAG for each new run (never reuses an existing folder)
RUN_BASE = "adv_no_uv_18_21_dt360"
# RUN_BASE = "adv_no_uv_23_dt360"  # old holdout 2023
# RUN_BASE = "adv_no_uv_13_22_dt360"  # old train 2013–2022
# RUN_BASE = "adv_no_uv_16_19_dt96"  # old dt96
RUN_TAG = "vgg_warm10_adv60_wd3e-4"  # default after wd ablation (was vgg_warm10_adv60)
# RUN_TAG = "vgg_warm10_adv60"  # old: no explicit wd in tag
# RUN_TAG = "lpips_warm10_adv60"  # old: LPIPS (alex) perceptual
RUN_NAME = f"{RUN_BASE}_{RUN_TAG}"
RUN_DIR = Path("Models/adv_no") / RUN_NAME

EPOCHS = 100
OPTIMIZER = "adamw"  # decoupled L2; use with weight_decay
WEIGHT_DECAY = 3e-4  # best arm in 30-ep ablation (0 / 1e-4 / 3e-4)

LOSS_MODE = "adv_no"  # "l1" = masked L1 only | "adv_no" = L1 + perceptual + adversarial
LOSS_METRIC = "components"  # masked L1 on u,v | "speed" = L1 on |U,V| instead

# --- spatial loss focus ---
LOSS_SPATIAL_MODE = "full"  # "full" = uniform over whole valid field | "gaussian_center" = center⊙G
LOSS_GAUSSIAN_SIGMA = 32.0  # only used when LOSS_SPATIAL_MODE == "gaussian_center"

CHANNEL_NAMES = ("u", "v")
iD = 2

# --- adv-NO: L1 + VGG perceptual + adversarial ---
BETA = 0.1
LAMBDA_PERC = 0.1
PERCEPTUAL_BACKEND = "vgg"  # multi-layer VGG-19 L1
PERCEPTUAL_WEIGHTS = (0.1, 0.1, 1.0, 1.0, 1.0)  # layer weights for VGG-19 features
# PERCEPTUAL_WEIGHTS = (1.0, 1.0, 1.0, 1.0, 1.0)  # equal layer weights (alternative)
LPIPS_NET = "alex"  # only used if PERCEPTUAL_BACKEND == "lpips"

# --- LPIPS perceptual (old) ---
# PERCEPTUAL_BACKEND = "lpips"
# LPIPS_NET = "alex"  # alex | vgg | squeeze

# Loss staging: L1 through ep 10, +VGG from ep 11, +adversarial from ep 31
PERCEPTUAL_WARMUP_EPOCHS = 10  # L1 only through epoch 10; VGG from epoch 11
WARMUP_EPOCHS = 30  # no adversarial through epoch 30; adversarial from epoch 31
# WARMUP_EPOCHS = 59  # old: adversarial from epoch 60

D_LR_RATIO = 0.5
D_SKIP_THRESHOLD = 0.2

USE_COSINE_LR = True

DETACHED = False
SHOW_INLINE_PLOTS = False
PREVIEW_LOSS_MASK = True  # save viz/loss_mask_preview.png before training
PROTECT_EXISTING_RUN = False  # allow same RUN_TAG; training overwrites prior results
DEVICE = "cuda:7"

VIZ_EVERY = 1
VIZ_SAMPLES = 1
VIZ_SEED = 42
VIZ_RESIDUAL_METRIC = "l1_smooth"  # error panels in saved PNGs only (loss unchanged)

VAL_SPLIT = 0.3
VAL_SPLIT_MODE = "random"
VAL_SPLIT_SEED = 42
VAL_RECONSTRUCTION_METRIC = "rmse"

print(f"Optimizer: {OPTIMIZER}, lr=2e-4, weight_decay={WEIGHT_DECAY}, epochs={EPOCHS}")
print(f"val: {int((1-VAL_SPLIT)*100)}% train / {int(VAL_SPLIT*100)}% val ({VAL_SPLIT_MODE}, seed={VAL_SPLIT_SEED}), metric={VAL_RECONSTRUCTION_METRIC}")

In [3]:
train_config = AdvNOTrainConfig(
    data_prefix=DATA_PREFIX,
    run_name=RUN_NAME,
    dt_hours=DT,
    epochs=EPOCHS,
    batch_size=4,
    lr=2e-4,
    optimizer=OPTIMIZER,
    weight_decay=WEIGHT_DECAY,
    d_lr=1e-4,
    d_lr_follow_g=True,
    d_lr_ratio=D_LR_RATIO,
    d_skip_threshold=D_SKIP_THRESHOLD,
    use_cosine_lr=USE_COSINE_LR,
    iD=iD,
    device=DEVICE,
    loss_mode=LOSS_MODE,
    loss_metric=LOSS_METRIC,
    loss_spatial_mode=LOSS_SPATIAL_MODE,
    loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
    protect_existing_run=PROTECT_EXISTING_RUN,
    preview_loss_mask=PREVIEW_LOSS_MASK,
    beta=BETA,
    lambda_perc=LAMBDA_PERC,
    perceptual_backend=PERCEPTUAL_BACKEND,
    lpips_net=LPIPS_NET,
    perceptual_weights=PERCEPTUAL_WEIGHTS,
    perceptual_warmup_epochs=PERCEPTUAL_WARMUP_EPOCHS,
    warmup_epochs=WARMUP_EPOCHS,
    checkpoint_subdir="Models/adv_no",
    isolate_runs=True,
    channel_names=CHANNEL_NAMES,
    viz_every=VIZ_EVERY,
    viz_samples=VIZ_SAMPLES,
    viz_seed=VIZ_SEED,
    viz_residual_metric=VIZ_RESIDUAL_METRIC,
    show_inline_plots=SHOW_INLINE_PLOTS,
    val_split=VAL_SPLIT,
    val_split_mode=VAL_SPLIT_MODE,
    val_split_seed=VAL_SPLIT_SEED,
    val_reconstruction_metric=VAL_RECONSTRUCTION_METRIC,
)

RUN_DIR.mkdir(parents=True, exist_ok=True)
config_path = RUN_DIR / f"train_config_{RUN_NAME}.json"
config_path.write_text(json.dumps(asdict(train_config), indent=2, default=str))
print(f"Run directory: {RUN_DIR}")
print(f"Wrote config:  {config_path}")
print(f"  data: {DATA_PREFIX} | iD={iD} | loss_metric={LOSS_METRIC}")
if LOSS_MODE == "l1":
    print("  loss mode: L1-only (no perceptual / discriminator)")
elif LOSS_MODE == "adv_no":
    perc_name = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"
    if PERCEPTUAL_WARMUP_EPOCHS == 0 and WARMUP_EPOCHS == 0:
        print(f"  loss mode: adv-NO — L1 + {perc_name} + adversarial from epoch 1")
    else:
        print(
            f"  loss mode: adv-NO — L1 through ep {PERCEPTUAL_WARMUP_EPOCHS}, "
            f"+{perc_name} from ep {PERCEPTUAL_WARMUP_EPOCHS + 1}, "
            f"+adv from ep {WARMUP_EPOCHS + 1}"
        )
else:
    print(f"  loss mode: {LOSS_MODE!r}")
if LOSS_SPATIAL_MODE == "full":
  spatial_note = "uniform over whole valid field"
elif LOSS_SPATIAL_MODE == "gaussian_center":
  spatial_note = f"gaussian_center (σ={LOSS_GAUSSIAN_SIGMA}) — state⊙G"
else:
  spatial_note = LOSS_SPATIAL_MODE
print(f"  loss spatial: {spatial_note}")

Run directory: Models/adv_no/adv_no_uv_23_dt360_vgg_warm10_adv60
Wrote config:  Models/adv_no/adv_no_uv_23_dt360_vgg_warm10_adv60/train_config_adv_no_uv_23_dt360_vgg_warm10_adv60.json
  data: data/sample_23_smallfixwin_uv_dt360 | iD=2 | loss_metric=components
  loss mode: adv-NO — L1 through ep 10, +VGG-19 from ep 11, +adv from ep 31
  loss spatial: uniform over whole valid field


In [4]:
from src.train.adv_no_train import _run_has_artifacts
from src.train.fix_window_viz import save_loss_mask_preview

if PROTECT_EXISTING_RUN and _run_has_artifacts(RUN_DIR, RUN_NAME):
    raise FileExistsError(
        f"Run already exists: {RUN_DIR}\n"
        "Change RUN_TAG to a new name before preview / training."
    )
elif _run_has_artifacts(RUN_DIR, RUN_NAME):
    print(f"Note: {RUN_DIR} exists — preview/training will overwrite prior results.")

if PREVIEW_LOSS_MASK:
    preview_path = save_loss_mask_preview(
        DATA_PREFIX,
        RUN_DIR / "viz" / "loss_mask_preview.png",
        sample_index=0,
        loss_spatial_mode=LOSS_SPATIAL_MODE,
        loss_gaussian_sigma=LOSS_GAUSSIAN_SIGMA,
        channel_names=CHANNEL_NAMES,
        show=SHOW_INLINE_PLOTS,
    )
    print(f"Loss mask preview: {preview_path}")
    if LOSS_SPATIAL_MODE == "full":
        print("  full mode: G=1 everywhere — loss treats all valid pixels equally")
    else:
        print(f"  gaussian_center: σ={LOSS_GAUSSIAN_SIGMA}px — center weighted more than edges")
else:
    print("Loss mask preview skipped (PREVIEW_LOSS_MASK=False)")

Loss mask preview: Models/adv_no/adv_no_uv_23_dt360_vgg_warm10_adv60/viz/loss_mask_preview.png
  full mode: G=1 everywhere — loss treats all valid pixels equally


In [5]:
log_dir = Path("logs")
log_dir.mkdir(exist_ok=True)
log_path = log_dir / f"train_{RUN_NAME}.log"

# --- background run (disabled for now) ---
# if DETACHED:
#     cmd = [sys.executable, "scripts/train_adv_no_fix_window.py", "--run-name", RUN_NAME]
#     with open(log_path, "w") as log_file:
#         proc = subprocess.Popen(
#             cmd,
#             stdout=log_file,
#             stderr=subprocess.STDOUT,
#             start_new_session=True,
#             cwd=Path.cwd(),
#         )
#     print(f"Started background adv-NO training PID {proc.pid}")
#     print(f"Log: {log_path}")
#     print(f"Monitor: tail -f {log_path}")
#     print("Stop with: kill", proc.pid)

# --- foreground: live notebook output + same log file; stop with Interrupt kernel ---
class _Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()


print(f"Foreground training — log: {log_path}")
print("Stop anytime: Kernel → Interrupt (or ■ button)\n")

_stdout = sys.stdout
with open(log_path, "w") as log_file:
    sys.stdout = _Tee(_stdout, log_file)
    try:
        run_adv_no_training(train_config)
    except KeyboardInterrupt:
        print("\nNotebook interrupt received — training stopped.")
    finally:
        sys.stdout = _stdout

print(f"\nLog saved: {log_path}")

Foreground training — log: logs/train_adv_no_uv_23_dt360_vgg_warm10_adv60.log
Stop anytime: Kernel → Interrupt (or ■ button)

Dataset: data/sample_23_smallfixwin_uv_dt360
Samples: 319 | crop 128 x 128


Using device: cuda:7
Loss mode: adv_no
Channels: iD=2 (u, v)
Reconstruction metric: components (uv L1)
Loss spatial focus: full valid field
lambda_perc: 0.1 | beta: 0.1
perceptual backend: VGG-19
perceptual_weights (VGG layers): (0.1, 0.1, 1.0, 1.0, 1.0)
Loss schedule: L1 through epoch 10, + perceptual from epoch 11, + adversarial from epoch 31
D LR: g×0.5 | D skip if loss < 0.2 | G scheduler: cosine
Checkpoints: Models/adv_no/adv_no_uv_23_dt360_vgg_warm10_adv60
History:     Models/adv_no/adv_no_uv_23_dt360_vgg_warm10_adv60/history/training_history_adv_no_uv_23_dt360_vgg_warm10_adv60.pt
Viz:         Models/adv_no/adv_no_uv_23_dt360_vgg_warm10_adv60/viz (every 1 epoch(s), fixed val indices [40])
Loss mask preview: Models/adv_no/adv_no_uv_23_dt360_vgg_warm10_adv60/viz/loss_mask_preview.png
epoch 1 | val uv L1 1.260850 | (improvement of inf)
Epoch 0001 [L1-warmup] | Train: 1.414075 (uv L1 1.414075, perc 0.000000, adv 0.000000, D 0.000000) | Val uv L1: 1.260850 (perc 0.573035) | G LR: 2.00

In [7]:
import json
import matplotlib.pyplot as plt
import torch

history_path = RUN_DIR / "history" / f"training_history_{RUN_NAME}.pt"
state_path = RUN_DIR / f"train_state_{RUN_NAME}.json"
log_path = Path(f"logs/train_{RUN_NAME}.log")
plot_path = Path("PNG") / f"adv_no_training_loss_curve_{RUN_NAME}.png"

if not history_path.exists():
    msg = [f"No history yet: {history_path}"]
    if state_path.exists():
        state = json.loads(state_path.read_text())
        msg.append(
            f"Training status: {state.get('status')} | "
            f"last completed epoch: {state.get('last_completed_epoch', 0)}/{state.get('epochs_planned', '?')}"
        )
    if log_path.exists():
        tail = log_path.read_text().splitlines()[-5:]
        msg.append("Recent log lines:")
        msg.extend(f"  {line}" for line in tail)
    msg.append("Re-run this cell after epoch 1 finishes (history is saved each epoch).")
    print("\n".join(msg))
else:
    hist = torch.load(history_path, map_location="cpu", weights_only=False)
    epochs = hist["epoch"]

    if not epochs:
        print("History file exists but is empty. Wait for epoch 1 to finish, then re-run.")
    else:
        recon_label = "speed L1" if LOSS_METRIC == "speed" else "uv L1"
        fig, ax = plt.subplots(1, 3, figsize=(14, 4))
        ax[0].plot(epochs, hist["train_l1"], label=f"Train {recon_label}")
        ax[0].plot(epochs, hist["val_l1"], label=f"Val {recon_label}")
        ax[0].set_title(f"Reconstruction ({recon_label})")
        ax[0].legend()
        ax[0].grid(True, alpha=0.3)

        use_adv_no = LOSS_MODE == "adv_no"
        perc_warmup = PERCEPTUAL_WARMUP_EPOCHS if use_adv_no else 0
        adv_warmup = WARMUP_EPOCHS if use_adv_no else 0

        perc_title = f"LPIPS ({LPIPS_NET})" if PERCEPTUAL_BACKEND == "lpips" else "VGG-19"
        if use_adv_no and any(hist.get("train_perceptual", [])):
            ax[1].plot(epochs, hist["train_perceptual"], label="Train perc")
            ax[1].plot(epochs, hist["val_perceptual"], label="Val perc")
            if perc_warmup > 0:
                ax[1].axvline(perc_warmup + 0.5, color="#2ca02c", linestyle=":", alpha=0.7)
            ax[1].set_title(f"Perceptual ({perc_title})")
            ax[1].legend()
            ax[1].grid(True, alpha=0.3)
        else:
            ax[1].set_title("Perceptual (adv-NO only)" if not use_adv_no else "Perceptual (starts after warmup)")
            ax[1].axis("off")

        if use_adv_no and any(hist.get("train_adv", [])):
            ax[2].plot(epochs, hist["train_adv"], label="Train adv")
            ax[2].plot(epochs, hist["train_d"], label="Train D")
            if adv_warmup > perc_warmup:
                ax[2].axvline(adv_warmup + 0.5, color="#888888", linestyle=":", alpha=0.7)
            ax[2].set_title("Adversarial terms")
            ax[2].legend()
            ax[2].grid(True, alpha=0.3)
        else:
            ax[2].set_title("Adversarial (adv-NO only)" if not use_adv_no else "Adversarial (starts after warmup)")
            ax[2].axis("off")

        plt.tight_layout()
        plot_path.parent.mkdir(exist_ok=True)
        plt.savefig(plot_path, dpi=150, bbox_inches="tight")
        plt.show()
        print(f"Best val {recon_label}: {hist['best_val_loss']:.6f} at epoch {hist['best_epoch']}")
        print(f"Saved plot: {plot_path}")

Best val uv L1: 0.659484 at epoch 62
Saved plot: PNG/adv_no_training_loss_curve_adv_no_uv_23_dt360_vgg_warm10_adv60.png
